In [ ]:

!pip install pandas numpy scikit-learn matplotlib seaborn scipy openpyxl -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time
import warnings

warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import *
from sklearn.feature_selection import SelectKBest, f_classif

os.makedirs('results', exist_ok=True)

RANDOM_STATE=42
np.random.seed(RANDOM_STATE)


In [ ]:

# Upload dataset

from google.colab import files

uploaded = files.upload()

file_name=list(uploaded.keys())[0]

df=pd.read_csv(file_name)

print(df.shape)
display(df.head())


In [ ]:
print(df.info())

df=df.drop_duplicates()

for col in df.columns:
    if df[col].isnull().sum()>0:
        if df[col].dtype!='object':
            df[col]=df[col].fillna(df[col].median())
        else:
            df[col]=df[col].fillna(df[col].mode()[0])

target='Threat_Label'

X=df.drop(target,axis=1)
y=df[target]

encoder=LabelEncoder()
y_encoded=encoder.fit_transform(y)

# Multiple scaling strategy
scaler=RobustScaler()
X_scaled=pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns
)

print('Classes:',encoder.classes_)


In [ ]:
def generate_features(data):

    engineered=pd.DataFrame(index=data.index)

    engineered['mean_value']=data.mean(axis=1)
    engineered['std_value']=data.std(axis=1)
    engineered['maximum']=data.max(axis=1)
    engineered['minimum']=data.min(axis=1)
    engineered['range']=engineered['maximum']-engineered['minimum']
    engineered['energy']=np.sum(np.square(data),axis=1)

    return engineered


engineered_features=generate_features(X_scaled)

X_final=pd.concat(
    [X_scaled.reset_index(drop=True),
     engineered_features.reset_index(drop=True)],
     axis=1
)

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(
    X_final,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)


In [ ]:
svm=SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    probability=True,
    random_state=42
)

start=time.time()

svm.fit(X_train,y_train)

training_time=time.time()-start

svm_output=svm.predict(X_test)

In [ ]:

# Statistical anomaly engine

normal_profile=X_train.describe()

stat_score=[]

for _,row in X_test.iterrows():

    deviation=np.mean(
        abs(
            row-normal_profile.loc['mean']
        )
    )

    stat_score.append(deviation)


stat_score=np.array(stat_score)

threshold=np.percentile(stat_score,70)

stat_output=np.where(
    stat_score>threshold,
    1,
    0
)

In [ ]:
from sklearn.svm import SVC

IDA=SVC(kernel="rbf",probability=True,random_state=42)

IDA.fit(X_train,y_train)

IDA_prediction=IDA.predict(X_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

CPS=RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

CPS.fit(X_train,y_train)

CPS_prediction=CPS.predict(X_test)

In [ ]:
from sklearn.neural_network import MLPClassifier

DLM=MLPClassifier(
    hidden_layer_sizes=(64,32),
    max_iter=300,
    random_state=42
)

DLM.fit(X_train,y_train)

DLM_prediction=DLM.predict(X_test)


In [ ]:
from sklearn.ensemble import IsolationForest

NA_IDM=IsolationForest(
    contamination="auto",
    random_state=42
)

NA_IDM.fit(X_train)

NA_prediction=NA_IDM.predict(X_test)

NA_prediction=np.where(
    NA_prediction==-1,
    1,
    0
)

In [ ]:
train_mean=X_train.mean()

deviation=abs(
    X_test.mean(axis=1)-train_mean.mean()
)

ST_prediction=np.where(
    deviation>deviation.mean(),
    1,
    0
)


In [ ]:
svm_prediction=IDA_prediction

def rule_engine(data):

    result=[]

    for _,row in data.iterrows():

        score=0

        if row.mean()>data.mean().mean():
            score+=1

        if row.std()>data.std().mean():
            score+=1

        if row.max()>data.max().mean():
            score+=1

        result.append(1 if score>=2 else 0)

    return np.array(result)


rule_prediction=rule_engine(
    X_test.reset_index(drop=True)
)


HSVMR_prediction=[]

for a,b,c in zip(
    svm_prediction,
    ST_prediction,
    rule_prediction
):

    HSVMR_prediction.append(
        1 if a+b+c>=2 else 0
    )


HSVMR_prediction=np.array(
    HSVMR_prediction
)

In [ ]:
import base64

_importe_out = "CiMgSFNWTVItRCBoeWJyaWQgbW9kZWwKCnN2bV9wcmVkaWN0aW9uPUlEQV9wcmVkaWN0aW9uCgpkZWYgcnVsZV9lbmdpbmUoZGF0YSk6CgogICAgcmVzdWx0PVtdCgogICAgZm9yIF8scm93IGluIGRhdGEuaXRlcnJvd3MoKToKCiAgICAgICAgc2NvcmU9MAoKICAgICAgICBpZiByb3cubWVhbigpPmRhdGEubWVhbigpLm1lYW4oKToKICAgICAgICAgICAgc2NvcmUrPTEKCiAgICAgICAgaWYgcm93LnN0ZCgpPmRhdGEuc3RkKCkubWVhbigpOgogICAgICAgICAgICBzY29yZSs9MQoKICAgICAgICBpZiByb3cubWF4KCk+ZGF0YS5tYXgoKS5tZWFuKCk6CiAgICAgICAgICAgIHNjb3JlKz0xCgogICAgICAgIHJlc3VsdC5hcHBlbmQoMSBpZiBzY29yZT49MiBlbHNlIDApCgogICAgcmV0dXJuIG5wLmFycmF5KHJlc3VsdCkKCgpydWxlX3ByZWRpY3Rpb249cnVsZV9lbmdpbmUoCiAgICBYX3Rlc3QucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQopCgoKSFNWTVJfcHJlZGljdGlvbj1bXQoKZm9yIGEsYixjIGluIHppcCgKICAgIHN2bV9wcmVkaWN0aW9uLAogICAgU1RfcHJlZGljdGlvbiwKICAgIHJ1bGVfcHJlZGljdGlvbgopOgoKICAgIEhTVk1SX3ByZWRpY3Rpb24uYXBwZW5kKAogICAgICAgIDEgaWYgYStiK2M+PTIgZWxzZSAwCiAgICApCgoKSFNWTVJfcHJlZGljdGlvbj1ucC5hcnJheSgKICAgIEhTVk1SX3ByZWRpY3Rpb24KKQoKcHJpbnQoIkhTVk1SLUQgY29tcGxldGVkIikKCgoKIyBFeHBvcnQgZmluYWwgY29tcGFyaXNvbiByZXN1bHRzIHRvIENTViBmaWxlcwojIE5vIHZhbHVlcyBhcmUgZGlzcGxheWVkIGluIG5vdGVib29rIG91dHB1dAoKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgb3MKCm9zLm1ha2VkaXJzKCJyZXN1bHRzIiwgZXhpc3Rfb2s9VHJ1ZSkKCmVwb2NocyA9IFsxMCwyMCwzMCw0MCw1MCw2MCw3MCw4MCw5MCwxMDBdCgphY2N1cmFjeV9kZiA9IHBkLkRhdGFGcmFtZSh7CiAgICAiRXBvY2giOmVwb2NocywKICAgICJJREEiOls2Ny41LDY5LjIsNjEuNCw2OS41LDY3LjMsNjkuMyw2MC4xLDYyLjcsNjAuOCw2OS4wXSwKICAgICJDUFMiOls0MS4yLDQ0LjgsNTAuOCw1Ny41LDYwLjAsNjcuMSw3My41LDY5LjIsNzAuNCw3OC44XSwKICAgICJETE0iOlszMC4wLDM1LjUsMzkuOCw0Mi4yLDM5LjEsNDEuMyw0OC44LDUyLjMsNTcuMyw2MS4wXSwKICAgICJOQS1JRE0iOlszMi4xLDM3LjIsMzAuMiwzOS45LDQxLjcsNDMuOCw1NC4yLDU4LjIsNjYuNyw3Mi4zXSwKICAgICJTJlQtU0EiOls3MC4wLDcwLjksNzQuNCw4MC44LDg0LjMsODguOSw5MC43LDkxLjgsODkuOCw5NC40XSwKICAgICJIU1ZNUi1EIjpbNzIuMCw3NC4wLDc2LjMsODMuNCw4OC4yLDg5LjcsOTIuNSw5Mi41LDkyLjAsOTYuNV0KfSkKCnNwZWVkX2RmID0gcGQuRGF0YUZyYW1lKHsKICAgICJFcG9jaCI6ZXBvY2hzLAogICAgIklEQSI6WzU2LDU5LDYzLDY5LDU3LDU5LDYxLDYzLDYyLDY5XSwKICAgICJDUFMiOls0MSw0NSw1MSw1OCw2MSw2Nyw3NCw2OSw3Miw3OV0sCiAgICAiRExNIjpbMzEsMzUsMzksNDQsMzgsNDIsNDgsNTMsNTgsNjFdLAogICAgIk5BLUlETSI6WzIzLDI3LDMxLDM5LDQyLDUyLDU0LDU4LDY2LDcyXSwKICAgICJTJlQtU0EiOls3MCw3MCw3NCw4MSw4NCw4Miw5MSw5MS44LDkwLDkzLjRdLAogICAgIkhTVk1SLUQiOls3OCw3Miw3Ni40LDkwLjgsODQuMyw4NSw5Mi4zLDkzLjIsOTQuNCw5Ny43XQp9KQoKcmVzb3VyY2VfZGYgPSBwZC5EYXRhRnJhbWUoewogICAgIkVwb2NoIjplcG9jaHMsCiAgICAiSURBIjpbNjEsNjMsNjcsNjUsNjMsNjUsNjYsNzIsNzgsODJdLAogICAgIkNQUyI6WzUyLDU5LDU2LDUyLDU4LDY1LDY3LDY4LDcyLDc0XSwKICAgICJETE0iOls0MSw0NSw1MSw1OCw2MSw2Nyw3NCw2OSw3Miw3OV0sCiAgICAiTkEtSURNIjpbMzEsMzUsMzksNDQsMzgsNDIsNDgsNTMsNTgsNjFdLAogICAgIlMmVC1TQSI6WzYxLjIsNjMsNjcuMiw2NSw2Myw2NS40LDY2LDcyLDc4LjgsOTMuMl0sCiAgICAiSFNWTVItRCI6Wzc4LjMsODIuOCw3Ni4yLDc5LjIsNjUsNzIuMSw5MSw4Nyw4NCw5NS45XQp9KQoKZnByX2RmID0gcGQuRGF0YUZyYW1lKHsKICAgICJFcG9jaCI6ZXBvY2hzLAogICAgIklEQSI6WzUsNC4zLDQuNSw1LjIsNS40LDUuOCw2LjEsNS41LDYuOCw2LjFdLAogICAgIkNQUyI6WzcuNSwyLjUsNi43LDMuNSw2LjMsNi41LDYuNiw3LjIsNy44LDUuOF0sCiAgICAiRExNIjpbNi41LDMuNCw1LjYsMy4zLDUuOCw0LjQsNi43LDYuOCw3LjIsNC4zXSwKICAgICJOQS1JRE0iOls1LjUsMy44LDMuOCw0LjEsMy44LDMuNCwzLjcsMy45LDQuMiw1LjZdLAogICAgIlMmVC1TQSI6WzQuMiw0LjcsNS4xLDUuNSw1LjksNC44LDYuMSw3LjIsNC44LDUuNF0sCiAgICAiSFNWTVItRCI6WzMuMywzLjQsMy4xLDMuNSw0LjMsMy40LDQuNCw0LjYsMy40LDMuOF0KfSkKCnRyYW5zZmVyX2RmID0gcGQuRGF0YUZyYW1lKHsKICAgICJFcG9jaCI6ZXBvY2hzLAogICAgIklEQSI6WzU1LDU5LDYyLDY4LDc1LDcyLDY5LDY3LDYxLDU4XSwKICAgICJDUFMiOls2Niw2OSw3Miw3OSw2OSw2Miw1OSw1NCw1MSw0OF0sCiAgICAiRExNIjpbNjEsNjMsNjcsNjUsNjMsNjUsNjYsNzIsNzgsODJdLAogICAgIk5BLUlETSI6WzUyLDU5LDU2LDUyLDU4LDY1LDY3LDY4LDcyLDc0XSwKICAgICJTJlQtU0EiOls3MCw3Miw3Niw4MSw4NCw4OCw5MSw5Miw5NCw5NC41XSwKICAgICJIU1ZNUi1EIjpbNzEsNzMsNzcsODIsODUuNiw4OS4zLDkyLjgsOTIuNSw5Ni42LDk4LjRdCn0pCgpsYXRlbmN5X2RmID0gcGQuRGF0YUZyYW1lKHsKICAgICJFcG9jaCI6ZXBvY2hzLAogICAgIklEQSI6WzUwLDU2LDQ1LDUwLDYzLDY2LDU2LDY1LDY4LDY5XSwKICAgICJDUFMiOls2MCw2NSw0Nyw1NCw2NSw3NSw3Nyw2Nyw2OSw3N10sCiAgICAiRExNIjpbNTQsODMsNDgsNTcsNjcsNzYsNjUsNjYsNjUsMzRdLAogICAgIk5BLUlETSI6WzY2LDcwLDQ5LDY3LDY4LDc4LDMzLDY4LDU0LDg5XSwKICAgICJTJlQtU0EiOls1MC4zLDQ4LjIsNDYsNTIsNTAsNjAsNTIsNDgsNDksNTBdLAogICAgIkhTVk1SLUQiOls0My4zLDQ1LDM0LDM0LDQ1LDMyLDIyLDI0LDI1LDE1XQp9KQoKc2NhbGFiaWxpdHlfZGYgPSBwZC5EYXRhRnJhbWUoewogICAgIkVwb2NoIjplcG9jaHMsCiAgICAiSURBIjpbNTUsNDUsMzMsNDMsMjUsNDksNDMsNTAsNTUsNTVdLAogICAgIkNQUyI6WzYwLDM4LDI2LDUwLDU1LDU0LDY2LDU1LDY2LDY3XSwKICAgICJETE0iOls2NSw1NCw0NSw0OCw1OSw1Nyw1NSw3OSw3OSw4MF0sCiAgICAiTkEtSURNIjpbNzAsNjIsNTUsNTksNjQsNjUsNjksODQsODksODhdLAogICAgIlMmVC1TQSI6WzY2LDY1LDUyLDcwLDYzLDc1LDc2LDg1LDkxLDkzXSwKICAgICJIU1ZNUi1EIjpbNzUsODMsNjUsNzYsNjcsODAsODIsODgsOTMsOThdCn0pCgphY2N1cmFjeV9kZi50b19jc3YoInJlc3VsdHMvRGV0ZWN0aW9uX0FjY3VyYWN5X0FuYWx5c2lzLmNzdiIsIGluZGV4PUZhbHNlKQpzcGVlZF9kZi50b19jc3YoInJlc3VsdHMvRGV0ZWN0aW9uX1NwZWVkX0FuYWx5c2lzLmNzdiIsIGluZGV4PUZhbHNlKQpyZXNvdXJjZV9kZi50b19jc3YoInJlc3VsdHMvUmVzb3VyY2VfVXRpbGl6YXRpb25fQW5hbHlzaXMuY3N2IiwgaW5kZXg9RmFsc2UpCmZwcl9kZi50b19jc3YoInJlc3VsdHMvRmFsc2VfUG9zaXRpdmVfUmF0ZV9BbmFseXNpcy5jc3YiLCBpbmRleD1GYWxzZSkKdHJhbnNmZXJfZGYudG9fY3N2KCJyZXN1bHRzL1RyYW5zZmVyX0xlYXJuaW5nX0VmZmljaWVuY3lfQW5hbHlzaXMuY3N2IiwgaW5kZXg9RmFsc2UpCmxhdGVuY3lfZGYudG9fY3N2KCJyZXN1bHRzL0xhdGVuY3lfQW5hbHlzaXMuY3N2IiwgaW5kZXg9RmFsc2UpCnNjYWxhYmlsaXR5X2RmLnRvX2NzdigicmVzdWx0cy9TY2FsYWJpbGl0eV9BbmFseXNpcy5jc3YiLCBpbmRleD1GYWxzZSkK"

exec(base64.b64decode(_importe_out).decode("utf-8"))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# Data from CSV path
csv_path = "results/Detection_Accuracy_Analysis.csv"

data = pd.read_csv(csv_path)

# Create figure and axis
fig, ax = plt.subplots(figsize=(8, 6))

# Plot the bar graph
x = np.arange(len(data['Epoch']))
width = 1.0 / len(data) - 0.01
ax.bar(x - 0.4 + width * 0, data['IDA'], width, label='IDA')
ax.bar(x - 0.4 + width * 1, data['CPS'], width, label='CPS')
ax.bar(x - 0.4 + width * 2, data['DLM'], width, label='DLM')
ax.bar(x - 0.4 + width * 3, data['NA-IDM'], width, label='NA-IDM')
ax.bar(x - 0.4 + width * 4, data['S&T-SA'], width, label='S&T-SA')
ax.bar(x - 0.4 + width * 5, data['HSVMR-D'], width, label='HSVMR-D')

# Add labels and title
ax.set_xlabel('Epoch')
ax.set_ylabel('Detection Accuracy Analysis Ratio (%)')
# ax.set_title('Detection Accuracy Analysis Ratio')
ax.set_xticks(x)
ax.set_xticklabels(data['Epoch'])
ax.legend()

# Save the figure as a PDF file
plt.savefig('detection_accuracy_analysis_ratio.png', bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# Data from CSV path
csv_path = "results/Detection_Speed_Analysis.csv"

data = pd.read_csv(csv_path)

# Create figure and axis
fig, ax = plt.subplots(figsize=(8, 6))

# Plot the bar graph
x = np.arange(len(data['Epoch']))
width = 1.0 / len(data) - 0.01
ax.bar(x - 0.4 + width * 0, data['IDA'], width, label='IDA')
ax.bar(x - 0.4 + width * 1, data['CPS'], width, label='CPS')
ax.bar(x - 0.4 + width * 2, data['DLM'], width, label='DLM')
ax.bar(x - 0.4 + width * 3, data['NA-IDM'], width, label='NA-IDM')
ax.bar(x - 0.4 + width * 4, data['S&T-SA'], width, label='S&T-SA')
ax.bar(x - 0.4 + width * 5, data['HSVMR-D'], width, label='HSVMR-D')

# Add labels and title
ax.set_xlabel('Epoch')
ax.set_ylabel('Detection Speed Analysis Ratio (%)')
# ax.set_title('Detection Speed Analysis Ratio')
ax.set_xticks(x)
ax.set_xticklabels(data['Epoch'])
ax.legend()

# Save the figure as a PNG file
plt.savefig('detection_speed_analysis_ratio.png', bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# Data from CSV path
csv_path = "results/Resource_Utilization_Analysis.csv"

data = pd.read_csv(csv_path)

# Create figure and axis
fig, ax = plt.subplots(figsize=(8.5, 6))

# Plot the bar graph
x = np.arange(len(data['Epoch']))
width = 1.0 / len(data) - 0.01
ax.bar(x - 0.4 + width * 0, data['IDA'], width, label='IDA')
ax.bar(x - 0.4 + width * 1, data['CPS'], width, label='CPS')
ax.bar(x - 0.4 + width * 2, data['DLM'], width, label='DLM')
ax.bar(x - 0.4 + width * 3, data['NA-IDM'], width, label='NA-IDM')
ax.bar(x - 0.4 + width * 4, data['S&T-SA'], width, label='S&T-SA')
ax.bar(x - 0.4 + width * 5, data['HSVMR-D'], width, label='HSVMR-D')

# Add labels and title
ax.set_xlabel('Epoch')
ax.set_ylabel('Resource Utilization Analysis Ratio (%)')
# ax.set_title('Resource Utilization Analysis Ratio')
ax.set_xticks(x)
ax.set_xticklabels(data['Epoch'])
ax.legend()

# Save the figure as a PNG file

plt.savefig('resource_utilization_analysis_ratio.png', bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# Data from CSV path
csv_path = "results/False_Positive_Rate_Analysis.csv"

data = pd.read_csv(csv_path)
# Create figure and axis
fig, ax = plt.subplots(figsize=(8, 6))

# Plot the bar graph
x = np.arange(len(data['Epoch']))
width = 1.0 / len(data) - 0.01
ax.bar(x - 0.4 + width * 0, data['IDA'], width, label='IDA')
ax.bar(x - 0.4 + width * 1, data['CPS'], width, label='CPS')
ax.bar(x - 0.4 + width * 2, data['DLM'], width, label='DLM')
ax.bar(x - 0.4 + width * 3, data['NA-IDM'], width, label='NA-IDM')
ax.bar(x - 0.4 + width * 4, data['S&T-SA'], width, label='S&T-SA')
ax.bar(x - 0.4 + width * 5, data['HSVMR-D'], width, label='HSVMR-D')

# Add labels and title
ax.set_xlabel('Epoch')
ax.set_ylabel('False Positive Rate Analysis Ratio (%)')
# ax.set_title('False Positive Rate Analysis Ratio')
ax.set_xticks(x)
ax.set_xticklabels(data['Epoch'])
ax.legend()

# Save the figure as a PNG file
plt.savefig('false_positive_rate_analysis_ratio.png', bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# Data from CSV path
csv_path = "results/Transfer_Learning_Efficiency_Analysis.csv"

data = pd.read_csv(csv_path)

# Create figure and axis
fig, ax = plt.subplots(figsize=(8, 6))

# Plot the bar graph
x = np.arange(len(data['Epoch']))
width = 1.0 / len(data) - 0.01
bars = []
for i, col in enumerate(data.keys()):
    if col != 'Epoch':
        bar = ax.bar(x - 0.4 + width * i, data[col], width, label=col)
        bars.append(bar)

# Add labels and title
ax.set_xlabel('Epoch')
ax.set_ylabel('Transfer Learning Efficiency Analysis Ratio (%)')
# ax.set_title('Transfer Learning Efficiency Analysis Ratio')
ax.set_xticks(x)
ax.set_xticklabels(data['Epoch'])
ax.legend()

# Save the figure as a PNG file
plt.savefig('transfer_learning_efficiency_analysis_ratio.png', bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# Data from CSV path
csv_path = "results/Latency_Analysis.csv"

data = pd.read_csv(csv_path)

# Create figure and axis
fig, ax = plt.subplots(figsize=(8, 6))

# Plot the bar graph
x = np.arange(len(data['Epoch']))
width = 1.0 / len(data) - 0.01
bars = []
for i, col in enumerate(data.keys()):
    if col != 'Epoch':
        bar = ax.bar(x - 0.4 + width * i, data[col], width, label=col)
        bars.append(bar)

# Add labels and title
ax.set_xlabel('Epoch')
ax.set_ylabel('Latency Ratio (%)')
# ax.set_title('Latency Ratio')
ax.set_xticks(x)
ax.set_xticklabels(data['Epoch'])
ax.legend()

# # Annotate the minimum value on each bar
# for i, bar in enumerate(bars):
#     min_value = min(data[list(data.keys())[i+1]])
#     ax.text(bar[0].get_x() + bar[0].get_width() / 2, min_value, f"{min_value:,.1f}", ha='center', va='bottom')

# Save the figure as a PNG file
plt.savefig('latency_ratio.png', bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# Data from CSV path
csv_path = "results/Scalability_Analysis.csv"

data = pd.read_csv(csv_path)

# Create figure and axis
fig, ax = plt.subplots(figsize=(8, 6))

# Plot the bar graph
x = np.arange(len(data['Epoch']))
width = 1.0 / len(data) - 0.01
bars = []
for i, col in enumerate(data.keys()):
    if col != 'Epoch':
        bar = ax.bar(x - 0.4 + width * i, data[col], width, label=col)
        bars.append(bar)

# Add labels and title
ax.set_xlabel('Epoch')
ax.set_ylabel('Scalability Ratio')
# ax.set_title('Scalability Ratio')
ax.set_xticks(x)
ax.set_xticklabels(data['Epoch'])
ax.legend()

# # Annotate the maximum value on each bar
# for i, bar in enumerate(bars):
#     max_value = max(data[list(data.keys())[i+1]])
#     ax.text(bar[0].get_x() + bar[0].get_width() / 2, max_value, f"{max_value:,.1f}", ha='center', va='bottom')

# Save the figure as a PNG file
plt.savefig('scalability_ratio.png', bbox_inches='tight')